### 과제 3

In [1]:
import pandas as pd
import numpy as np
import os

# --- 경로 설정 ---
# VENDOR_ANALYSIS_PATH와 GICS_PATH가 올바른지 확인해주세요.
VENDOR_ANALYSIS_PATH = "../output/problem2_vendor/problem2_vendor_analysis_base.csv"
GICS_PATH = "../output/gics_clean.csv" 
OUT_DIR = "../output/problem3_sensitivity" 
os.makedirs(OUT_DIR, exist_ok=True)

GICS_LEVELS = ['sector', 'industry_group', 'industry', 'sub_industry']

# --- 1. 데이터 로드 및 시그널 필터링 ---
print("-> 1. 데이터 로드 및 시그널 필터링 시작...")

try:
    df_analysis = pd.read_csv(
        VENDOR_ANALYSIS_PATH, 
        usecols=['symbol', 'surprise_z', 'return_post_1d', 'return_post_2d'],
        dtype={'surprise_z': np.float32, 'return_post_1d': np.float32, 'return_post_2d': np.float32}
    ).dropna(subset=['surprise_z', 'return_post_1d'])

    MEAN_Z_SCORE = df_analysis['surprise_z'].mean() 
    STD_Z_SCORE = df_analysis['surprise_z'].std() 
    
    # Positive Signal인 행만 필터링 (Z > +2σ)
    is_positive_signal = df_analysis['surprise_z'] > MEAN_Z_SCORE + 2 * STD_Z_SCORE
    df_positive_reaction = df_analysis[is_positive_signal].copy()
    
    if df_positive_reaction.empty:
        print("분석: Positive Signal 데이터가 없어 산업별 민감도 분석을 건너뜁니다.")
        exit()

    df_gics = pd.read_csv(GICS_PATH, dtype={c: np.float16 for c in GICS_LEVELS})
    
    # GICS와 Positive Signal 데이터 병합
    df_merged = pd.merge(df_positive_reaction, df_gics, on='symbol', how='left')
    df_merged = df_merged.dropna(subset=['sector', 'return_post_1d'])
    
    print(f"✅ 데이터 통합 및 필터링 완료. 분석 대상 데이터 수: {len(df_merged)}개")

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다. 경로를 확인하세요: {e}")
    exit()
except Exception as e:
    print(f"❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: {e}")
    exit()


# --- 2. 산업별 민감도 분석 (그룹핑 및 출력) ---

print(f"➡️ GICS '{GICS_LEVELS[0]}' 레벨별 민감도 분석 시작...")

# 그룹핑 및 평균 수익률 계산
df_gics_sensitivity = df_merged.groupby(GICS_LEVELS[0])[['return_post_1d', 'return_post_2d']].mean()
df_gics_sensitivity = df_gics_sensitivity.mul(100).round(3) 
df_gics_sensitivity = df_gics_sensitivity.rename(columns={'return_post_1d': 'Avg_Return_Post_1D (%)', 'return_post_2d': 'Avg_Return_Post_2D (%)'})


# --- 3. 최종 결과 출력 및 저장 ---
OUT_ANALYSIS_FILE = os.path.join(OUT_DIR, "problem3_gics_sensitivity.csv")
df_gics_sensitivity.to_csv(OUT_ANALYSIS_FILE)

print(f"\n[OK] 산업별 민감도 분석 결과 저장 완료: {OUT_ANALYSIS_FILE}")

print("\n" + "="*70)
print(f"과제 3: Positive Signal (Long)에 대한 GICS '{GICS_LEVELS[0]}' 민감도 분석 (Post 1D)")
print("="*70)

df_gics_sensitivity_pos_sorted = df_gics_sensitivity.sort_values(by='Avg_Return_Post_1D (%)', ascending=False)
print(df_gics_sensitivity_pos_sorted.to_markdown())
print("\n" + "="*70)

if not df_gics_sensitivity_pos_sorted.empty:
    most_sensitive_gics = df_gics_sensitivity_pos_sorted.index[0]
    max_return = df_gics_sensitivity_pos_sorted.iloc[0]['Avg_Return_Post_1D (%)']
    print(f"가장 민감한 산업 ({GICS_LEVELS[0]}): {most_sensitive_gics} (평균 {max_return:.3f}%)")
else:
    print("분석 결과가 비어있습니다.")
print("="*70)

-> 1. 데이터 로드 및 시그널 필터링 시작...
✅ 데이터 통합 및 필터링 완료. 분석 대상 데이터 수: 299개
➡️ GICS 'sector' 레벨별 민감도 분석 시작...


/Users/masterj/Documents/GitHub/team5/StockPlay-Data-analysis/5mil/lib/python3.12/site-packages/pandas/io/parsers/c_parser_wrapper.py:234: RuntimeWarning: overflow encountered in cast
  chunks = self._reader.read_low_memory(nrows)


AssertionError: 

In [3]:
import pandas as pd
import numpy as np
import os

# --- 경로 설정 ---
VENDOR_ANALYSIS_PATH = "../output/problem2_vendor/problem2_vendor_analysis_base.csv"
GICS_PATH = "../output/gics_clean.csv" 
OUT_DIR = "../output/problem3_sensitivity" 
os.makedirs(OUT_DIR, exist_ok=True)

GICS_LEVELS = ['sector', 'industry_group', 'industry', 'sub_industry']

print("➡️ 1. 데이터 로드 및 시그널 필터링 시작...")

try:
    # 1) 과제 2 분석 데이터 로드 (메모리 최적화)
    df_analysis = pd.read_csv(
        VENDOR_ANALYSIS_PATH, 
        usecols=['symbol', 'surprise_z', 'return_post_1d', 'return_post_2d'],
        dtype={'surprise_z': np.float32, 'return_post_1d': np.float32, 'return_post_2d': np.float32}
    ).dropna(subset=['surprise_z', 'return_post_1d'])

    MEAN_Z_SCORE = df_analysis['surprise_z'].mean() 
    STD_Z_SCORE = df_analysis['surprise_z'].std() 
    
    is_positive_signal = df_analysis['surprise_z'] > MEAN_Z_SCORE + 2 * STD_Z_SCORE
    df_positive_reaction = df_analysis[is_positive_signal].copy()
    
    if df_positive_reaction.empty:
        print("분석: Positive Signal 데이터가 없어 산업별 민감도 분석을 건너뜁니다.")
        exit()

    # 2) GICS 산업 분류 데이터 로드 및 오류 수정
    df_gics = pd.read_csv(GICS_PATH)
    
    # !!! 오류 해결: 그룹 키로 사용할 GICS 컬럼을 float64로 강제 캐스팅 !!!
    for col in GICS_LEVELS:
        if col in df_gics.columns:
            df_gics[col] = df_gics[col].astype(np.float64) 
    # -------------------------------------------------------------
    
    # 3) GICS와 Positive Signal 데이터 병합
    df_merged = pd.merge(df_positive_reaction, df_gics, on='symbol', how='left')
    df_merged = df_merged.dropna(subset=['sector', 'return_post_1d'])
    
    print(f"✅ 데이터 통합 및 필터링 완료. 분석 대상 데이터 수: {len(df_merged)}개")

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다. 경로를 확인하세요: {e}")
    exit()
except Exception as e:
    print(f"❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: {e}")
    exit()


# --- 3. 산업별 민감도 분석 (그룹핑) ---

# GICS 'sector' 기준으로 수익률의 평균을 계산
print(f"➡️ GICS '{GICS_LEVELS[0]}' 레벨별 민감도 분석 시작...")

# 그룹 연산 전에 수익률 컬럼 타입을 float64로 캐스팅하여 Pandas 내부 Assertion Error 방지 (이전 단계의 오류 방지)
df_merged['return_post_1d'] = df_merged['return_post_1d'].astype(np.float64)
df_merged['return_post_2d'] = df_merged['return_post_2d'].astype(np.float64)


df_gics_sensitivity = df_merged.groupby(GICS_LEVELS[0])[['return_post_1d', 'return_post_2d']].mean()
df_gics_sensitivity = df_gics_sensitivity.mul(100).round(3) 
df_gics_sensitivity = df_gics_sensitivity.rename(columns={'return_post_1d': 'Avg_Return_Post_1D (%)', 'return_post_2d': 'Avg_Return_Post_2D (%)'})


# --- 4. 최종 결과 출력 및 저장 ---

OUT_ANALYSIS_FILE = os.path.join(OUT_DIR, "problem3_gics_sensitivity.csv")
df_gics_sensitivity.to_csv(OUT_ANALYSIS_FILE)

print(f"\n[OK] 산업별 민감도 분석 결과 저장 완료: {OUT_ANALYSIS_FILE}")

print("\n" + "="*70)
print(f"과제 3: Positive Signal (Long)에 대한 GICS '{GICS_LEVELS[0]}' 민감도 분석 (Post 1D)")
print("="*70)

df_gics_sensitivity_pos_sorted = df_gics_sensitivity.sort_values(by='Avg_Return_Post_1D (%)', ascending=False)
print(df_gics_sensitivity_pos_sorted.to_markdown())
print("\n" + "="*70)

if not df_gics_sensitivity_pos_sorted.empty:
    most_sensitive_gics = df_gics_sensitivity_pos_sorted.index[0]
    max_return = df_gics_sensitivity_pos_sorted.iloc[0]['Avg_Return_Post_1D (%)']
    print(f"가장 민감한 산업 ({GICS_LEVELS[0]}): {most_sensitive_gics} (평균 {max_return:.3f}%)")
else:
    print("분석 결과가 비어있습니다.")
print("="*70)

➡️ 1. 데이터 로드 및 시그널 필터링 시작...
✅ 데이터 통합 및 필터링 완료. 분석 대상 데이터 수: 299개
➡️ GICS 'sector' 레벨별 민감도 분석 시작...

[OK] 산업별 민감도 분석 결과 저장 완료: ../output/problem3_sensitivity/problem3_gics_sensitivity.csv

과제 3: Positive Signal (Long)에 대한 GICS 'sector' 민감도 분석 (Post 1D)
|   sector |   Avg_Return_Post_1D (%) |   Avg_Return_Post_2D (%) |
|---------:|-------------------------:|-------------------------:|
|       30 |                    0.776 |                    2.159 |
|       20 |                    0.631 |                    0.757 |
|       35 |                    0.623 |                    0.763 |
|       15 |                    0.441 |                    0.75  |
|       45 |                    0.151 |                    0.807 |
|       25 |                   -0.738 |                    0.594 |
|       10 |                   -1.538 |                   -0.35  |

가장 민감한 산업 (sector): 30.0 (평균 0.776%)
